Python script that processes other datasets for annotation.

In [ ]:
from dataset_processing import BIODIVNER_DIR, BIODIVNER_LABELS, biodivner_process_bio_documents
from dataset_processing import IBMCCNER_DIR, IBMCCNER_LABELS, ibmccner_process_bio_documents 
from dataset_processing import convert_to_labelstudio_format

from datasets import load_dataset

import json

## Loading BioDivNER

Using a modified `load_biodivner()` function from `dataset_processing`:

In [ ]:
split = ["train", "test", "dev"]
print(f"Loading '{BIODIVNER_DIR}' with splits: {split}")

if type("string") == type(split):
    split = [split]
    
print("Converting from BIO tags to char spans.")
biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=BIODIVNER_DIR, 
    labels_to_keep=BIODIVNER_LABELS, 
    split=split
    )

for i in range(len(biodivner_structured_data)):
    biodivner_structured_data[i]["sentence"] = biodivner_structured_data[i]["text"]
    biodivner_structured_data[i]["sentence_id"] = i
    
ls_biodivner = convert_to_labelstudio_format(biodivner_structured_data, "0")

with open("RESULTS/BioDivNER/biodivner_full.json", "w", encoding="utf-8") as outfile:
    json.dump(ls_biodivner, outfile, indent=4)

## Loading IBMCCNER


Using a modified `load_ibmccner()` function from `dataset_processing`:

In [ ]:
# 1. Load and pre-process the data into a document-wise list of lines
split = ["train", "validation", "test"]
print(f"Loading '{IBMCCNER_DIR}' with splits: {split}")
ds = load_dataset(IBMCCNER_DIR) 
train_documentwise = []
temp_list = []
if type(split) == type(["list"]):
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith('-DOCSTART-'):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
else:
    for line in ds[split]["text"]:
        if line.strip().startswith('-DOCSTART-'):
            if temp_list:
                train_documentwise.append(temp_list)
            temp_list = []
        temp_list.append(line)
    if temp_list:
        train_documentwise.append(temp_list)

print("Converting from BIO tags to char spans.")        
structured_documents_char_spans = ibmccner_process_bio_documents(
    document_list=train_documentwise,
    labels_to_keep=IBMCCNER_LABELS
)

for i in range(len(structured_documents_char_spans)):
    structured_documents_char_spans[i]["sentence"] = structured_documents_char_spans[i]["text"]
    structured_documents_char_spans[i]["sentence_id"] = i

ls_ibmccner = convert_to_labelstudio_format(structured_documents_char_spans, "0")

with open("RESULTS/IBMCCNER/ibmccner_full.json", "w", encoding="utf-8") as outfile:
    json.dump(ls_ibmccner, outfile, indent=4)

## Load ClimateIE

In [ ]:
import os
import re
# Use uppercase for constants as per Python conventions.
DATA_DIRECTORY = "DATA/ClimateIE/human_corpus/"
# A set of expected entity labels.
EXPECTED_LABELS = {
    'experiment', 'instrument', 'location',
    'model', 'natural hazard', 'ocean circulation',
    'teleconnection', 'variable', 'weather event',
    'provider', 'project', 'event'
}


try:
    file_names = [f for f in os.listdir(DATA_DIRECTORY) if f.endswith('.json')]
except FileNotFoundError:
    print(f"Error: The directory '{DATA_DIRECTORY}' was not found.")
    file_names = []

# Initialize a dictionary to hold a set for each label.
# Using a dictionary comprehension is the correct and pythonic way.
# Using a set() will ensure that each entity substring is stored only once per category.
entities_by_label_ClimateIE = {label: set() for label in EXPECTED_LABELS}

# Process each file.
for file_name in file_names:
    # Use os.path.join for safe path construction across different operating systems.
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    
    with open(file_path, "r", encoding="utf-8") as f:
        document = json.load(f)

    for heading in re.split(r"<heading>[\w\s\.]+</heading>", document["text"]):
        print(heading.strip())
        print("----"*20)
    print(document.keys())
    print(len(document["text"]))
    # print(document["text"])
    
    print(document["entities"]["95745"].keys())
    break
    # # Safely get the entities dictionary.
    # entities = document.get("entities", {})
    
    # # Iterate through each entity's data in the "entities" dictionary.
    # for entity_data in entities.values():
    #     label = entity_data.get("label")
    #     substring = entity_data.get("substring")
        
    #     # Check if the label is one we expect and add the substring to the corresponding set.
    #     if label in entities_by_label_ClimateIE:
    #         entities_by_label_ClimateIE[label].add(substring)

In [ ]:
biodivner_structured_data[0].keys()